# ✨ YAPAY ZEKA GELİŞİM KAMPI | Senior Seviye: **Fine-Tuning** (Hafta 3)

🎖️ **HAFTA 3: Model Özelleştirme – Fine-Tuning, PEFT, LoRA ve Veri Hazırlığı**

# 📘 Fine-Tuning: Modeli Kendi Dünyana Uyarlamak
## Supervised Fine-Tuning (SFT) – PEFT – LoRA – Dataset Formatting

1. ve 2. haftalarda modellerin nasıl çalıştığını ve hazır modellerin nasıl kullanıldığını gördük. Ancak genel amaçlı modeller (LLaMA, GPT vb.) bazen sizin özel veriniz, kurum içi jargonunuz veya spesifik bir görev (örn: sadece SQL yazması) için yeterli olmayabilir. 

Bu hafta, milyarlarca parametreli devasa modelleri, devasa donanımlara ihtiyaç duymadan nasıl kendi verimizle "akıllandıracağımızı" öğreneceğiz.

# 1️⃣ Fine-Tuning Türleri
Modelleri eğitirken temelde iki yol izleriz:

- **Full Fine-Tuning:** Modelin tüm ağırlıklarını yeniden güncelleriz. Çok yüksek GPU gücü (VRAM) gerektirir.
- **Parameter-Efficient Fine-Tuning (PEFT):** Modelin ana ağırlıklarını dondurup, sadece küçük bir grup parametreyi eğitiriz. 
  - *Avantajı:* Çok daha az bellek tüketir, eğitim daha hızlıdır ve modelin ana yeteneklerini unutmasını (Catastrophic Forgetting) engeller.

# 2️⃣ LoRA (Low-Rank Adaptation) Nedir?
Günümüzde Senior seviyesinde en çok kullanılan PEFT tekniğidir. 
- Modelin devasa matrislerini eğitmek yerine, bu matrislerin yanına çok daha küçük "adaptör" matrisleri ekleriz.
- Eğitim sırasında sadece bu küçük matrisler güncellenir.
- Sonuçta elimizde MB'larca boyutta küçük bir "adaptör" dosyası olur. Bu dosyayı ana modele takıp çıkarabiliriz.

# 3️⃣ Instruction Tuning (Talimat Eğitimi)
LLM'lerin bir asistana dönüşmesini sağlayan süreçtir. Veri seti genellikle şu formatta hazırlanır:
- **Instruction:** Kullanıcının verdiği görev.
- **Input (Opsiyonel):** Görevin bağlamı.
- **Output:** Modelden beklenen ideal cevap.

Örnek (JSONL formatı):
`{"instruction": "Aşağıdaki metni özetle.", "input": "...", "output": "..."}`

In [ ]:
# Gerekli kütüphaneler
# !pip install -q -U bitsandbytes transformers peft accelerate trl

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 1. Modeli 4-bit yükleyerek belleği optimize edelim
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

# 2. LoRA Konfigürasyonu
lora_config = LoraConfig(
    r=8,              # Rank: Matrisin ne kadar küçük olacağı (8-16-32 idealdir)
    lora_alpha=32,    # Ölçeklendirme faktörü
    target_modules=["q_proj", "v_proj"], # Modelin hangi katmanlarına odaklanacağı
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. PEFT Modelini Oluşturma
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # Kaç parametre eğiteceğimizi görelim

# 4️⃣ SFTTrainer (Supervised Fine-tuning Trainer)
Hugging Face'in `trl` kütüphanesi, model eğitme sürecini inanılmaz kolaylaştırır. Veri setini veririz, hiperparametreleri ayarlarız ve eğitim başlar.

**Önemli Hiperparametreler:**
- **Learning Rate:** Öğrenme hızı (Fine-tuning'de çok düşük seçilir, örn: 2e-4).
- **Epoch:** Veri setinin üzerinden kaç tur geçileceği.
- **Batch Size:** Her adımda kaç örneğin işleneceği.

# 5️⃣ QLoRA: En Verimli Fine-Tuning Tekniği
2. haftada Quantization (Niceleme) görmüştük. **QLoRA (Quantized LoRA)**, modelin ağırlıklarını 4-bit olarak dondurup, üzerine LoRA adaptörlerini ekleyerek eğitme tekniğidir. 

**Neden QLoRA?**
- Normalde 7B parametreli bir modeli eğitmek için 40GB+ VRAM gerekirken, QLoRA ile bu işlem 12-16GB VRAM (ücretsiz Colab seviyesi) ile yapılabilir.
- Modelin ana ağırlıkları 4-bit olduğu için hafıza kullanımı minimumdadır.

# 6️⃣ Chat Templates ve Tokenizasyonun Önemi
Modelin sadece ham metni alması yetmez. Modern modeller (Llama-3, Mistral vb.) konuşmanın nerede başlayıp nerede bittiğini anlamak için özel "template" yapıları kullanır. 

Örnek bir **ChatML** formatı:
```text
<|im_start|>user
Nasılsın?<|im_end|>
<|im_start|>assistant
İyiyim, sana nasıl yardımcı olabilirim?<|im_end|>

In [ ]:
# Veriyi modelin şablonuna (Template) uygun hale getirme örneği
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

chat = [
  {"role": "user", "content": "Bana Python'da bir dekoratör yaz."},
  {"role": "assistant", "content": "Tabii, işte basit bir dekoratör örneği..."}
]

# Modeli eğitirken bu fonksiyonu kullanarak veriyi 'formatlıyoruz'
formatted_chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
print("Modelin Göreceği Ham Format:\n", formatted_chat)

# 7️⃣ SFTTrainer: Eğitim Sürecini Yönetmek
`SFTTrainer`, eğitim döngüsünü (backpropagation, loss calculation, logging) otomatize eder. Ancak parametreleri doğru ayarlamak "Senior" dokunuşudur.

**Kritik Parametreler:**
- **Gradient Accumulation Steps:** Eğer GPU belleğiniz çok küçükse, batch'leri biriktirip ağırlıkları birkaç adımda bir güncelleyerek "sanki büyük GPU'nuz varmış gibi" davranmanızı sağlar.
- **Mixed Precision (bf16/fp16):** Eğitimi hızlandırmak ve bellekten tasarruf etmek için kullanılır.
- **Max Sequence Length:** Verinizin model tarafından kesilmemesi (truncation) için kritik değerdir.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Senior Dokunuşu: Eğitim argümanlarını optimize edelim
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,     # Her adımda 4 örnek
    gradient_accumulation_steps=4,      # 4 adımda bir ağırlık güncelle (Etkin Batch Size = 16)
    learning_rate=2e-4,                 # Fine-tuning için ideal düşük hız
    logging_steps=10,
    num_train_epochs=1,
    max_steps=100,                      # Test amaçlı sınırlama
    fp16=True,                          # Bellek tasarrufu ve hız için mixed precision
    optim="paged_adamw_32bit",          # QLoRA için özel optimize edilmiş optimizer
    report_to="none"                    # İleride 'wandb' olarak değiştirilebilir
)

# Trainer kurulumu (Veri seti ve modelin hazır olduğu varsayılmıştır)
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=dataset,             # Hazırladığınız instruction veri seti
#     peft_config=lora_config,           # Yukarıda tanımladığımız LoRA ayarları
#     dataset_text_field="text",
#     max_seq_length=512,
#     tokenizer=tokenizer,
#     args=training_args,
# )

# trainer.train() # Eğitimi başlatan komut

# 8️⃣ Eğitim İzleme: Weights & Biases (W&B)
Eğitimin nasıl gittiğini sadece terminalden izlemek senior bir yaklaşım değildir. Loss grafiği, GPU kullanımı ve örnek çıktıları görselleştirmek için **W&B** veya **MLflow** gibi araçlar kullanılır.

Bu araçlarla:
- Eğitim patlamalarını (loss spike) anında görürsünüz.
- Farklı deneyleri (farklı learning rate vb.) birbiriyle karşılaştırabilirsiniz.

# 9️⃣ Model Merging: Adaptörü Ana Modele Bağlamak
Eğitim bittiğinde elimizde sadece küçük bir **LoRA adaptörü** olur. Bu adaptörü tek başına kullanamayız. Uygulamaya (production) çıkmadan önce bu adaptörü ana model (base model) ile birleştirmemiz gerekir.

Bu işleme **Merging** denir. İşlem sonunda elinizde tek bir `model_weights` dosyası olur ve artık herhangi bir adaptöre ihtiyaç duymadan modelinizi yayına alabilirsiniz.

In [ ]:
# Eğitim bittikten sonra adaptörü birleştirme (Mantıksal Akış)
# model = model.merge_and_unload()
# model.save_pretrained("my_final_model")
# tokenizer.save_pretrained("my_final_model")

print("Model başarıyla birleştirildi ve yayına hazır!")

## Mutlaka İzle :

- **QLoRA - Efficient Finetuning Explained:** https://youtu.be/TPcXVJ1VSRI
- **Training LLMs with SFTTrainer & TRL:** https://youtu.be/7p-Vp0o8hAc
- **Model Merging Techniques:** https://youtu.be/p6fP8NfE2k8
- **Weights & Biases Intro for AI:** https://youtu.be/C-fGvId9u70